# Week 6 Day 3: Context Engineering with MCP  
People used to talk about Prompt Engineering, but now it's given way to a new Skill "Context Engineering".  
Philipp Schmid of Google DeepMind wrote the seminal post about Context Engineering.  
https://www.philschmid.de/context-engineering 
In following labs, we will put Context Engineering into practice, heavily using MCP servers.  
1. Long-term memory: a knowledge graph the agent writes to and reads back.
2. Web search: fresh information from thelive web.
3. Agentic RAG: a vector store the agent fills from its own research, then searches.
4. Integrations: connecting to a live external service, with a local fallback.

In [2]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio, create_static_tool_filter
import os
from pathlib import Path
from datetime import datetime
from IPython.display import Markdown, display
load_dotenv(override=True)

True

In [3]:
# On Windows, a stdio MCP server started from a Jupyter kernel writes to a stderr stream with no
# real file descriptor and crashes with io.UnsupportedOperation: fileno. We send the server's
# stderr to the null device so it always has somewhere real to write, which lets every cell below
# use MCPServerStdio exactly as the OpenAI Agents SDK documents it. Mac and Linux are unaffected.
import functools
import subprocess
import agents.mcp.server

agents.mcp.server.stdio_client = functools.partial(agents.mcp.server.stdio_client, errlog=subprocess.DEVNULL)

## Part 1: Long-term Memory  
Our first context source is memory. This is the official knowledge graph server: it stores entities, observations about them, and the relationships between them, and keeps them on disk between runs. We point it at `memory/memory.json`, so that we can open it and read the graph it builds.  
That gives an agent something it normally lacks: a memory that outlives the conversation. The agent writes facts as it learns them and reads them back later.  

https://github.com/modelcontextprotocol/servers/tree/main/src/memory

In [4]:
memory_path = os.path.abspath("memory/memory.json")
memory_params = {'command': 'npx', 'args' : ['-y', '@modelcontextprotocol/server-memory'], 'env' : {'MEMORY_FILE_PATH' : memory_path}}

async with MCPServerStdio(params=memory_params, client_session_timeout_seconds=60) as server:
    memory_tools = await server.list_tools()

memory_tools

[Tool(name='create_entities', title='Create Entities', description='Create multiple new entities in the knowledge graph', inputSchema={'$schema': 'http://json-schema.org/draft-07/schema#', 'type': 'object', 'properties': {'entities': {'type': 'array', 'items': {'type': 'object', 'properties': {'name': {'type': 'string', 'description': 'The name of the entity'}, 'entityType': {'type': 'string', 'description': 'The type of the entity'}, 'observations': {'type': 'array', 'items': {'type': 'string'}, 'description': 'An array of observation contents associated with the entity'}}, 'required': ['name', 'entityType', 'observations']}}}, 'required': ['entities']}, outputSchema={'$schema': 'http://json-schema.org/draft-07/schema#', 'type': 'object', 'properties': {'entities': {'type': 'array', 'items': {'type': 'object', 'properties': {'name': {'type': 'string', 'description': 'The name of the entity'}, 'entityType': {'type': 'string', 'description': 'The type of the entity'}, 'observations': {'

In [5]:
instructions = 'You use your entity tools as a persisten memory to store and recall information about your conversations.'
request = "My name is fa. I am an LLM engineer. I am teaching a course about AI agents, including the incredible MCP protocol. MCP is a protocol for connecting agents with tools, resources and prompt templates, and makes it easy to integrate AI agents with capabilities."
model = 'gpt-4o-mini'

In [6]:
async with MCPServerStdio(params=memory_params, client_session_timeout_seconds=60) as mcp_server:
    agent = Agent(name='agent', instructions = instructions, model = model, mcp_servers=[mcp_server])
    with trace('conversation'):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

I've noted your name and profession along with details about your course and the MCP protocol. If you'd like to add more observations or make any updates, just let me know!

In [7]:
async with MCPServerStdio(params= memory_params, client_session_timeout_seconds= 60) as mcp_server:
    agent = Agent(name = 'agent', instructions = instructions, model = model, mcp_servers = [mcp_server])
    with trace('conversation'):
        result = await Runner.run(agent, "My name is Fa, What do you know about me?")
    display(Markdown(result.final_output))

I found some information about you, Fa:

- You are an LLM engineer.
- You are teaching a course about AI agents.
- Your course includes the MCP protocol.

If there's anything more specific you'd like to add or update, let me know!

### Check out the Trace   
https://platform.openai.com/traces

## Part 2: Web Search   
A model only knows what it was trained on. Web search is the context source that keeps it current.  
We use Tavily, a search API built for agents: it returns clean, ranked results an LLM can use directly. Tavily maintains its own MCP server,
so connecting is a one-line.  
This needs a free API key:
1. Sign up at https://www.tavily.com
2. The free tier gives you 1,000 searches a month, with no credit card.
3. Copy AP key (it starts with `tvly-`) and add it to `.env` file:  
`TAVILY_APIKEY = tvly-xxxx`

In [8]:
tavily_params = {
    'command' : 'npx',
    'args' : ['-y', 'tavily-mcp@latest'],
    'env' : {
        'TAVILY_API_KEY' : os.getenv('TAVILY_API_KEY')
    }
}

async with MCPServerStdio(
    params=tavily_params,
    client_session_timeout_seconds = 60
) as server:
    tavily_tools = await  server.list_tools()

tavily_tools

[Tool(name='tavily_search', title=None, description='Search the web for current information on any topic. Use for news, facts, or data beyond your knowledge cutoff. Returns snippets and source URLs.', inputSchema={'type': 'object', 'properties': {'query': {'type': 'string', 'description': 'Search query'}, 'search_depth': {'type': 'string', 'enum': ['basic', 'advanced', 'fast', 'ultra-fast'], 'description': "The depth of the search. 'basic' for generic results, 'advanced' for more thorough search, 'fast' for optimized low latency with high relevance, 'ultra-fast' for prioritizing latency above all else", 'default': 'basic'}, 'topic': {'type': 'string', 'enum': ['general'], 'description': 'The category of the search. This will determine which of our agents will be used for the search', 'default': 'general'}, 'time_range': {'type': 'string', 'description': 'The time range back from the current date to include in the search results', 'enum': ['day', 'week', 'month', 'year']}, 'start_date':

Tavily's server offers several tools (search, extract, crawl, map, research). For this lab we only want plain web search, so we restrict the server to `tavily_search`. The OpenAI Agents SDK lets you hand an agent just the tools you choose with a static tool filter. Curating an agent's tools like this is itself context engineering.

In [9]:
instructions = "You search the web for information and briefly summarize the takeways."
request = f"Please research the latest news on Amazon stock price and briefly summarize its outlook. FOr context, the current date is {datetime.now().strftime('%Y-%m-%d')}"
model = 'gpt-4o-mini'
search_only = create_static_tool_filter(allowed_tool_names = ['tavily_search'])

In [11]:
async with MCPServerStdio(params = tavily_params, client_session_timeout_seconds = 60, tool_filter = search_only) as mcp_server:
    agent = Agent(name = 'agent', instructions = instructions, model = model, mcp_servers = [mcp_server])
    with trace('conversation'):
        result = await Runner.run(agent, request)

    display(Markdown(result.final_output))

As of mid-July 2026, Amazon’s stock (AMZN) has shown a positive trajectory, particularly driven by significant growth in its Amazon Web Services (AWS) segment and overall better-than-expected earnings reports.

### Key Takeaways:

1. **Stock Performance**:
   - As of July 14, AMZN's stock price is approximately $247.04, reflecting a **6.9% increase** over the past week.
   - Year-to-date, the stock is up about **5.13%**, but has seen a **5.4% decline** over the past month.

2. **AWS Growth**:
   - AWS is a major growth driver, reporting a **28% revenue growth** in Q1 2026, marking its fastest pace in 15 quarters.
   - Analysts are optimistic about AWS' continued growth, with a **backlog of $364 billion** enhancing future prospects.

3. **Financial Outlook**:
   - For Q2 2026, Amazon projects revenue between **$194 billion and $199 billion**, indicating a possible **16% to 19% growth**.
   - Operating income is expected to range between **$20 billion to $24 billion**, a solid performance compared to last year.

4. **Investments and Challenges**:
   - Amazon plans to borrow **$25 billion** to support its data center expansion, as its capital expenditures currently exceed operational cash flow.
   - Concerns around free cash flow remain, with a reported decrease to **$1.2 billion** over the trailing twelve months.

5. **Analyst Sentiment**:
   - Despite some challenges, analysts show strong confidence, with **66 rating the stock as Buy or higher**, targeting a price of **$312**.

### Conclusion:
Overall, Amazon's stock outlook appears favorable, driven by AWS growth and solid earnings forecasts, despite current cash flow pressures and heavy investment needs. Investors are keenly awaiting the next earnings report on July 30 for further insights into these trends.

## Part 3: Agentic RAG  
RAG, retrieval augmented generation, means giving a model relevant documents to ground its answer. The usual setup loads documents  into a vector store up front. Agentic RAG turns that around: the agent builds the knowledge base itself, deciding what is worth keeping and storing it as it works.  
We used the official Qdrant MCP server. Qdrant is a vector database, and the server exposes two tools: one stores a piece of text, the other finds the most relevant stored text with a local model, so there is no extra API key. The first store or search downloads that small embedding model, so it pauses once on first use.  

https://github.com/qdrant/mcp-server-qdrant  

We hand one agent both Tavily and Qdrant: it researches a topic on the web, stores what it learns, then answers from its own knowledge base.


In [12]:
vectordb_path = Path("memory/qdrant")
vectorstore_params = {
    'command' : 'uvx',
    'args' : ['mcp-server-qdrant'],
    'env' : {
        "QDRANT_LOCAL_PATH" : str(vectordb_path),
        "COLLECTION_NAME" : 'knowledge'
    },
}

async with MCPServerStdio(params = vectorstore_params, 
                          client_session_timeout_seconds=120 ) as server:
    vectorstore_tools = await server.list_tools()

vectorstore_tools

[Tool(name='qdrant-find', title=None, description='Look up memories in Qdrant. Use this tool when you need to: \n - Find memories by their content \n - Access memories for further analysis \n - Get some personal information about the user', inputSchema={'properties': {'query': {'description': 'What to search for', 'title': 'Query', 'type': 'string'}}, 'required': ['query'], 'type': 'object'}, outputSchema=None, icons=None, annotations=None, meta=None, execution=None),
 Tool(name='qdrant-store', title=None, description='Keep the memory for later use, when you are asked to remember something.', inputSchema={'properties': {'information': {'description': 'Text to store', 'title': 'Information', 'type': 'string'}, 'metadata': {'anyOf': [{'additionalProperties': True, 'type': 'object'}, {'type': 'null'}], 'default': None, 'description': 'Extra metadata stored along with memorised information. Any json is accepted.', 'title': 'Metadata'}}, 'required': ['information'], 'type': 'object'}, outpu

In [14]:
INSTRUCTIONS = """
You research topic on the web and build up a knowledge base for later. 
When you learn something worth keeping, store it in your knowledge base.
When you are asked what you know, search your knowledge base and asnwer from it."""

model = 'gpt-4o-mini'

async with MCPServerStdio(params = tavily_params, client_session_timeout_seconds=60, tool_filter = search_only) as search_server:
    async with MCPServerStdio(params = vectorstore_params, client_session_timeout_seconds=120) as vector_server:
        agent = Agent(name= 'researcher', instructions = INSTRUCTIONS, model = model, mcp_servers = [search_server, vector_server])
        with trace('conversation'):
            result = await Runner.run(agent, "Research the latest news on Nvidia and store the key facts in your knowledge base.", max_turns = 20)
        display(Markdown(result.final_output))

I've stored key facts about Nvidia's latest news in the knowledge base:

### Recent Nvidia News Highlights:
1. **AI in Space (March 19, 2026)**: Nvidia announced plans to bring AI and accelerated computing to space.
   
2. **Tokenomics for AI (March 17, 2026)**: CEO Jensen Huang discussed 'tokenomics' as a new currency for recruitment and productivity in AI at the GTC conference.

3. **Blackwell GPU Launch (March 18, 2024)**: Nvidia introduced its Blackwell architecture, featuring a dual die chiplet design with high data transfer rates of 10 terabytes per second.

4. **Partnership Expansion (March 19, 2024)**: The company extended collaborations with major cloud providers like AWS, Google Cloud, and Microsoft Azure.

5. **AI Factories in Saudi Arabia (May 13, 2025)**: A partnership with HUMAIN aims to establish AI factories in Saudi Arabia.

6. **Revenue Growth (August 27, 2025)**: Nvidia reported a revenue of $46.7 billion for Q2, with data center revenue at $41.1 billion, reflecting a 56% year-over-year increase.

If you need more information or updates, feel free to ask!

The agent below has only the knowledge base, no web search. Whatever it tells us about Nvidia, it is recalling from waht the first agent stored. That is the retrieval half of RAG.

In [15]:
async with MCPServerStdio(params=vectorstore_params, client_session_timeout_seconds=120) as vector_server:
    agent  = Agent(name = 'researcher', instructions = INSTRUCTIONS, model = model, mcp_servers = [vector_server])
    with trace("retriever"):
        result = await Runner.run(agent, "Based on your knowledge base, what's the latest on Nvidia")
        
    display(Markdown(result.final_output))

Here are the latest highlights regarding Nvidia:

1. **AI in Space (March 19, 2026)**: Nvidia plans to bring AI and accelerated computing to space, tracking a broader trend among tech companies.

2. **Tokenomics for AI (March 17, 2026)**: In a keynote at the GTC conference, CEO Jensen Huang introduced the concept of "tokenomics" as a new currency for recruitment and productivity in the AI sector.

3. **Blackwell GPU Launch (March 18, 2024)**: Nvidia launched its Blackwell architecture, which features a dual die chiplet design and promises high data transfer rates of 10 terabytes per second.

4. **Partnership Expansion (March 19, 2024)**: The company has extended collaborations with major cloud providers, including AWS, Google Cloud, and Microsoft Azure, to integrate its latest GPUs.

5. **AI Factories in Saudi Arabia (May 13, 2025)**: Nvidia partnered with HUMAIN to establish AI factories, which aims to position Saudi Arabia as a global leader in AI.

6. **Revenue Growth (August 27, 2025)**: Nvidia reported a revenue of $46.7 billion for Q2, with data center revenue at $41.1 billion, reflecting a 56% year-over-year increase.

If you need more detailed information on any of these points, feel free to ask!

### Check out the trace

https://platform.openai.com/traces

## Part 4: Integrations  
The last context source is a live external service. Here that is market data from Massive (formerly Polygon.io), a popular financial data provider that publishes its own MCP server.  
Setting this up is optional. Massive offers a free API key with no credit card, giving you their real end of day market data:  
1. Sign up at https://www.massive.com
2. Create an API key.
3. Add it to your `.env` file.  
`MASSIVE_API_KEY = xxxx`  

If massive api is not setup, the next cell falls back to a local market server, the same kind of MCP server we built on Day 2, which servers simulated prices so the rest of the lab still works.

In [16]:
massive_api_key = os.getenv("MASSIVE_API_KEY")

if massive_api_key:
    market_params = {
        'command' : 'uvx',
        'args' : ["--from", 'git+https://github.com/massive-com/mcp_massive@v0.10.0', 'mcp_massive'],
        'env' : {"MASSIVE_API_KEY" : massive_api_key}
    }
else:
    market_params = {'command' : 'uv',
                     'args' : ['run', '-m', 'backend.market_server']}
    
async with MCPServerStdio(params = market_params, client_session_timeout_seconds=120) as server:
    market_tools = await server.list_tools()

market_tools

[Tool(name='lookup_share_price', title=None, description='This tool provides the current price of the given stock symbol.\n\n    Args:\n        symbol: the symbol of the stock\n    ', inputSchema={'properties': {'symbol': {'title': 'Symbol', 'type': 'string'}}, 'required': ['symbol'], 'title': 'lookup_share_priceArguments', 'type': 'object'}, outputSchema={'properties': {'result': {'title': 'Result', 'type': 'number'}}, 'required': ['result'], 'title': 'lookup_share_priceOutput', 'type': 'object'}, icons=None, annotations=None, meta=None, execution=None)]

In [17]:
instructions = "You answer questions about the stock market."
request = 'What was the most recent price that SpaceX (SPCX) traded at?'
model = 'gpt-4o-mini'

async with MCPServerStdio(params = market_params, client_session_timeout_seconds=120) as mcp_server:
    agent = Agent(name = 'agent', instructions = instructions, model = model, mcp_servers = [mcp_server])
    with trace('conversation'):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

The most recent price that SpaceX (SPCX) traded at is $139.14.